# Сегментация карт ориентирования — обучение и тест (CUDA)

Notebook для Linux-сервера с GPU. Логика лежит в `src/`; здесь — оркестрация.

**Перед стартом на сервере**
1. Залить репозиторий и папку `dataset/` (+ `reports/class_balance.json`).
2. Поставить зависимости из `requirements-cloud.txt` (не полный `requirements.txt`).
3. **Restart kernel**, затем ячейки сверху вниз.

> На DataSphere/облачных образах уже есть `torch`+CUDA и Jupyter.  
> Сообщения pip про конфликты с `tensorflow` / `fastai` / `google-*` можно игнорировать.

In [ ]:
# 0) Корень проекта и зависимости
from pathlib import Path
import sys
import os

# Если notebook открыт не из корня репозитория — поправьте ROOT вручную.
ROOT = Path.cwd()
if not (ROOT / "src").exists():
    # notebook лежит в notebooks/
    if (ROOT.parent / "src").exists():
        ROOT = ROOT.parent
    else:
        raise FileNotFoundError("Не найден каталог src/. Укажите ROOT вручную.")

os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT =", ROOT)
print("cwd  =", Path.cwd())

In [ ]:
# Установка (один раз). После неё: Kernel → Restart, затем ячейки с начала.

# ВАЖНО для облачного Jupyter (DataSphere и т.п.):
# - НЕ обновляйте torch (уже есть CUDA-сборка в образе).
# - НЕ ставьте jupyter/ipywidgets.
# - Используйте requirements-cloud.txt (мягкие пины, без numpy 2 / protobuf 7).

# Если уже успели поставить тяжёлый requirements.txt и окружение «поехало» —
# раскомментируйте блок rollback ниже, затем cloud-install.

# --- rollback к совместимым версиям (опционально) ---
# %pip install -U "numpy<1.27" "protobuf<5" "opencv-python-headless<4.11" "Pillow<11" "scipy<1.14"

# --- рекомендуемая установка ---
%pip install -r requirements-cloud.txt

print("Готово. Сделайте Kernel → Restart перед следующей ячейкой.")

In [ ]:
# 1) Проверка импортов + CUDA (после Restart kernel)
import importlib
import torch

required = [
    "segmentation_models_pytorch",
    "timm",
    "transformers",
    "albumentations",
    "cv2",
    "yaml",
    "tqdm",
    "PIL",
]
missing = []
for name in required:
    try:
        importlib.import_module(name)
        print(f"OK  {name}")
    except Exception as e:
        missing.append(name)
        print(f"FAIL {name}: {e}")

if missing:
    raise ImportError(f"Не установлены: {missing}. Запустите ячейку установки и Restart kernel.")

from src.utils.device import cuda_info, get_device
from src.utils.seed import set_seed

info = cuda_info()
for k, v in info.items():
    print(f"{k}: {v}")

device = get_device()
assert device.type == "cuda", (
    "CUDA недоступна. Не переустанавливайте torch из PyPI без CUDA index — "
    "на облаке обычно уже есть torch+cu118/cu12."
)
torch.backends.cudnn.benchmark = True
x = torch.zeros(1, device=device)
print("OK →", device, torch.cuda.get_device_name(0), "tensor", x.dtype)

In [ ]:
# 2) Конфиг эксперимента
from copy import deepcopy
from src.utils.config import load_experiment_config, deep_update

# Варианты: configs/baseline_unet.yaml | configs/quality_segformer.yaml | configs/smoke_cpu.yaml
CONFIG_PATH = "configs/baseline_unet.yaml"

cfg = load_experiment_config(CONFIG_PATH)

# Опциональные оверрайды под вашу GPU (пример для ~16–24 GB):
overrides = {
    "train": {
        # "batch_size": 8,          # уменьшить при OOM
        # "grad_accumulation": 2,   # компенсировать маленький batch
        # "epochs": 100,
        # "resume": None,           # или путь к last.pt
    },
    "data": {
        # "num_workers": 8,
    },
}
cfg = deep_update(cfg, overrides)

print("experiment:", cfg.get("experiment_name"))
print("model:", cfg["model"])
print("batch_size:", cfg["train"]["batch_size"], "amp:", cfg["train"].get("amp"))
print("dataset_root:", cfg["data"]["dataset_root"])
print("index_csv:", cfg["data"]["index_csv"])

In [ ]:
# 3) Проверка данных: split + overlay
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

from src.data.dataset import load_index, create_dataloaders
from src.data.labels import channels_to_index, build_priority_indices, overlay_prediction, colorize_label

set_seed(int(cfg.get("seed", 42)))

train_rec = load_index(cfg["data"]["index_csv"], cfg["data"]["dataset_root"], split="train", kinds=["tile"])
val_rec = load_index(cfg["data"]["index_csv"], cfg["data"]["dataset_root"], split="val", kinds=["tile"])
test_rec = load_index(cfg["data"]["index_csv"], cfg["data"]["dataset_root"], split="test", kinds=["tile"])
print(f"tiles: train={len(train_rec)} val={len(val_rec)} test={len(test_rec)}")
print(f"num_classes={len(cfg['classes'])} model.classes={cfg['model']['classes']}")

# сверка channels shape + свёртка на первом тайле
priority = build_priority_indices(cfg["label_priority"], cfg["classes"])
rec = train_rec[0]
channels = np.load(rec.channels)
assert channels.shape[0] == len(cfg["classes"]), channels.shape
mask = channels_to_index(channels, priority)
print(f"tile {rec.map_name}/{rec.sample}: channels={channels.shape} unique={np.unique(mask).tolist()}")

# визуализация тайла (image.jpg + channels.npy)
img = np.array(Image.open(rec.image).convert("RGB"))
fig, ax = plt.subplots(1, 3, figsize=(12, 4))
ax[0].imshow(img); ax[0].set_title("image"); ax[0].axis("off")
ax[1].imshow(colorize_label(mask)); ax[1].set_title("label"); ax[1].axis("off")
ax[2].imshow(overlay_prediction(img, mask)); ax[2].set_title("overlay"); ax[2].axis("off")
plt.tight_layout(); plt.show()


In [ ]:
# 4) Веса классов (effective number + optional boost)
from src.losses import compute_class_weights

boost = cfg["loss"].get("class_weight_boost") or {}
weights = compute_class_weights(
    class_balance_json=cfg["data"]["class_balance_json"],
    classes=cfg["classes"],
    empty_names=cfg.get("empty_classes", []),
    mode=cfg["loss"].get("class_weight_mode", "effective_number"),
    beta=float(cfg["loss"].get("effective_beta", 0.9999)),
    clip_min=float(cfg["loss"].get("weight_clip_min", 0.25)),
    clip_max=float(cfg["loss"].get("weight_clip_max", 10.0)),
    empty_weight=float(cfg["loss"].get("empty_class_weight", 0.0)),
    num_classes=int(cfg["model"]["classes"]),
    class_weight_boost={str(k): float(v) for k, v in boost.items()},
)
if boost:
    print("boost:", boost)
for i, name in sorted(cfg["classes"].items()):
    mark = " *" if name in boost else ""
    print(f"{i:2d} {name:20s} {weights[i].item():.4f}{mark}")


In [ ]:
# 5) Sanity: DataLoader + forward на GPU
from src.models import build_model

# временно меньше workers для быстрой проверки в notebook
_cfg = deepcopy(cfg)
_cfg["data"]["num_workers"] = min(4, int(cfg["data"].get("num_workers", 4)))
_cfg["train"]["batch_size"] = min(4, int(cfg["train"]["batch_size"]))

train_loader, val_loader, test_loader = create_dataloaders(_cfg)
batch = next(iter(train_loader))
print("image", tuple(batch["image"].shape), "mask", tuple(batch["mask"].shape))

model = build_model(cfg).to(device)
with torch.no_grad():
    logits = model(batch["image"].to(device))
print("logits", tuple(logits.shape), "on", device)
del model, logits
torch.cuda.empty_cache()
print("forward OK")

## Обучение

Ячейка ниже вызывает `src.train.train(cfg)`. Чекпоинты: `checkpoints/<experiment>/best.pt` и `last.pt`.  
Логи TensorBoard: `runs/<experiment>/tb`.

In [ ]:
# 6) TRAIN — запуск / продолжение обучения
from src.train import train

# --- продолжить с чекпоинта ---
# last.pt — последний шаг (модель + optimizer + ema); best.pt — лучший mIoU
CKPT_RESUME = Path(cfg["train"]["checkpoint_dir"]) / cfg["experiment_name"] / "last.pt"
# CKPT_RESUME = Path(cfg["train"]["checkpoint_dir"]) / cfg["experiment_name"] / "best.pt"

if CKPT_RESUME.exists():
    cfg["train"]["resume"] = str(CKPT_RESUME)
    print("resume ←", CKPT_RESUME)
else:
    cfg["train"]["resume"] = None
    print("чекпоинт не найден — обучение с нуля")

# epochs должно быть больше, чем уже пройдено (например, было 11 → поставьте 100+)
# cfg["train"]["epochs"] = 100

result = train(cfg)
print(result)

## Оценка и инференс


In [ ]:
# 7) Оценка на тайлах (val / test)
from src.evaluate import evaluate_tiles

CKPT = Path(cfg["train"]["checkpoint_dir"]) / cfg["experiment_name"] / "best.pt"
assert CKPT.exists(), f"Нет чекпоинта: {CKPT} — сначала обучите модель"

tile_metrics = evaluate_tiles(cfg, str(CKPT), split="val")
print(f"val mIoU={tile_metrics['mIoU']:.4f} mDice={tile_metrics['mDice']:.4f}")
print("critical IoU:")
for k, v in sorted(tile_metrics.get("critical_iou", {}).items()):
    print(f"  {k:20s} {v:.4f}")

In [ ]:
# 8) Инференс на тайле test-карты + визуализация
# (full-рендеров в датасете больше нет — только tiles/*/image.jpg)
from src.infer_fullmap import load_model_from_checkpoint
import torch
from src.postprocess import morphological_close_linear, export_cost_map
from src.data.transforms import build_eval_transforms

from src.evaluate import list_full_maps
test_maps = list_full_maps(cfg, split="test")
print("test maps:", test_maps[:10], ("..." if len(test_maps) > 10 else ""))
MAP_NAME = test_maps[0]

test_tiles = [r for r in test_rec if r.map_name == MAP_NAME]
rec = test_tiles[0]
print("tile:", rec.map_name, rec.sample, rec.image)

image = np.array(Image.open(rec.image).convert("RGB"))
gt = channels_to_index(np.load(rec.channels), priority)
print("tile size:", image.shape)

model = load_model_from_checkpoint(CKPT, cfg, device)
tfm = build_eval_transforms()
batch = tfm(image=image, mask=gt)
img_t = batch["image"].unsqueeze(0).to(device)
with torch.no_grad():
    pred = model(img_t).argmax(1).cpu().numpy()[0].astype(np.uint8)
pred_pp = morphological_close_linear(pred, cfg)

out_dir = Path(cfg["train"]["runs_dir"]) / cfg["experiment_name"] / "notebook_infer"
out_dir.mkdir(parents=True, exist_ok=True)
stem = f"{MAP_NAME}_{rec.sample}"
Image.fromarray(pred).save(out_dir / f"{stem}_label.png")
Image.fromarray(overlay_prediction(image, pred)).save(out_dir / f"{stem}_overlay.png")
np.save(out_dir / f"{stem}_cost.npy", export_cost_map(pred_pp, cfg))
print("saved →", out_dir)

fig, ax = plt.subplots(1, 3, figsize=(14, 5))
ax[0].imshow(image); ax[0].set_title(stem); ax[0].axis("off")
ax[1].imshow(overlay_prediction(image, gt)); ax[1].set_title("GT overlay"); ax[1].axis("off")
ax[2].imshow(overlay_prediction(image, pred)); ax[2].set_title("pred overlay"); ax[2].axis("off")
plt.tight_layout(); plt.show()


## Послойный просмотр предсказания

Для каждого класса — отдельная бинарная маска (и опционально сравнение с GT).
Нужны переменные `image`, `pred`, `MAP_NAME`, `rec` из ячейки инференса выше.


In [ ]:
# 8b) Послойные маски по классам (pred / GT / overlay на тайле)
from src.data.labels import CLASS_COLORS

SHOW_EMPTY = False       # True — показывать и классы без пикселей
SHOW_BACKGROUND = False  # обычно не интересен
COMPARE_GT = True        # GT из channels.npy тайла
COLS = 3                 # колонок в сетке (на класс: pred[+gt][+overlay])

assert "pred" in dir() and "image" in dir(), "Сначала выполните ячейку инференса (pred, image)"

# даунскейл на всякий случай
scale = max(1, max(image.shape[:2]) // 1200)
img_v = image[::scale, ::scale]
pred_v = pred[::scale, ::scale]

gt_v = None
if COMPARE_GT:
    gt_full = channels_to_index(np.load(rec.channels), priority)
    gt_v = gt_full[::scale, ::scale]

classes_sorted = sorted(cfg["classes"].items())  # (idx, name)
panels = []
for idx, name in classes_sorted:
    if not SHOW_BACKGROUND and idx == 0:
        continue
    pred_mask = pred_v == idx
    n_pred = int(pred_mask.sum())
    n_gt = int((gt_v == idx).sum()) if gt_v is not None else None
    if not SHOW_EMPTY and n_pred == 0 and (n_gt is None or n_gt == 0):
        continue

    color = np.array(CLASS_COLORS[idx] if idx < len(CLASS_COLORS) else (255, 0, 255), dtype=np.uint8)
    pred_rgb = np.zeros_like(img_v)
    pred_rgb[pred_mask] = color

    row = [("pred " + name, pred_rgb)]
    if gt_v is not None:
        gt_rgb = np.zeros_like(img_v)
        gt_rgb[gt_v == idx] = color
        row.append(("gt " + name, gt_rgb))
    overlay = img_v.copy()
    overlay[pred_mask] = (0.45 * overlay[pred_mask] + 0.55 * color).astype(np.uint8)
    row.append(("on map", overlay))
    panels.append(row)

n_rows = len(panels)
n_cols = max(len(r) for r in panels) if panels else 1
fig, axes = plt.subplots(n_rows, n_cols, figsize=(4 * n_cols, 3.2 * max(n_rows, 1)))
if n_rows == 1:
    axes = np.array([axes])
axes = np.atleast_2d(axes)
for i, row in enumerate(panels):
    for j in range(n_cols):
        ax = axes[i, j]
        if j < len(row):
            title, arr = row[j]
            ax.imshow(arr)
            ax.set_title(title, fontsize=9)
        ax.axis("off")
plt.tight_layout(); plt.show()
print(f"shown {len(panels)} classes for {MAP_NAME}/{rec.sample}")


## Confusion по классу `road`

Куда «утекает» road: среди пикселей GT=`road` — распределение pred; среди pred=`road` — распределение GT.  
Нужен `CKPT` (ячейка 7) или укажите путь вручную.

In [ ]:
# 8c) Confusion вокруг road (val tiles)
from collections import Counter

from src.data.dataset import create_dataloaders
from src.data.labels import class_name_to_index
from src.infer_fullmap import load_model_from_checkpoint
from tqdm.auto import tqdm

FOCUS = "road"  # можно "track_path" / "impassable"
TOP_K = 10

if "CKPT" not in dir() or not Path(CKPT).exists():
    CKPT = Path(cfg["train"]["checkpoint_dir"]) / cfg["experiment_name"] / "best.pt"
assert Path(CKPT).exists(), f"Нет чекпоинта: {CKPT}"

name_to_idx = class_name_to_index(cfg["classes"])
idx_to_name = {i: n for n, i in name_to_idx.items()}
focus_idx = name_to_idx[FOCUS]

_cfg = deepcopy(cfg)
_cfg["data"]["num_workers"] = min(4, int(cfg["data"].get("num_workers", 4)))
_cfg["train"]["batch_size"] = min(8, int(cfg["train"]["batch_size"]))
_, val_loader, _ = create_dataloaders(_cfg)

model = load_model_from_checkpoint(CKPT, cfg, device)
# gt_class -> pred_class counts (только где gt==FOCUS или pred==FOCUS)
gt_to_pred = Counter()   # для пикселей GT=FOCUS: куда предсказали
pred_to_gt = Counter()   # для пикселей pred=FOCUS: что было в GT
n_gt = n_pred = n_tp = 0

model.eval()
with torch.no_grad():
    for batch in tqdm(val_loader, desc=f"confusion:{FOCUS}"):
        images = batch["image"].to(device, non_blocking=True)
        masks = batch["mask"].cpu().numpy().reshape(-1)
        preds = model(images).argmax(1).cpu().numpy().reshape(-1)

        gt_m = masks == focus_idx
        pr_m = preds == focus_idx
        n_gt += int(gt_m.sum())
        n_pred += int(pr_m.sum())
        n_tp += int((gt_m & pr_m).sum())

        if gt_m.any():
            gt_to_pred.update(preds[gt_m].tolist())
        if pr_m.any():
            pred_to_gt.update(masks[pr_m].tolist())

iou = n_tp / max(n_gt + n_pred - n_tp, 1)
print(f"\n{FOCUS}: pixels_gt={n_gt}  pixels_pred={n_pred}  TP={n_tp}  IoU={iou:.4f}")
print(f"ckpt: {CKPT}\n")

print(f"GT={FOCUS} → pred (куда утекает, top {TOP_K}):")
for cls, cnt in gt_to_pred.most_common(TOP_K):
    print(f"  {idx_to_name.get(cls, cls):20s}  {cnt:10d}  ({100 * cnt / max(n_gt, 1):5.1f}%)")

print(f"\npred={FOCUS} ← GT (ложные срабатывания, top {TOP_K}):")
for cls, cnt in pred_to_gt.most_common(TOP_K):
    print(f"  {idx_to_name.get(cls, cls):20s}  {cnt:10d}  ({100 * cnt / max(n_pred, 1):5.1f}%)")

# барчарт
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
items = gt_to_pred.most_common(TOP_K)
ax[0].barh([idx_to_name.get(i, i) for i, _ in items][::-1], [c for _, c in items][::-1], color="#c44")
ax[0].set_title(f"GT={FOCUS} → pred")
ax[0].set_xlabel("pixels")
items = pred_to_gt.most_common(TOP_K)
ax[1].barh([idx_to_name.get(i, i) for i, _ in items][::-1], [c for _, c in items][::-1], color="#48a")
ax[1].set_title(f"pred={FOCUS} ← GT")
ax[1].set_xlabel("pixels")
plt.tight_layout()
plt.show()


In [ ]:
# 9) (Опционально) Оценка с агрегацией по картам (тайлы grouped by map)
from src.evaluate import evaluate_fullmaps

full_metrics = evaluate_fullmaps(cfg, str(CKPT), split="test")
print(f"per-map aggregate mIoU={full_metrics['mIoU']:.4f}")
for name, m in list(full_metrics.get("per_map", {}).items())[:10]:
    print(f"  {name:40s} mIoU={m['mIoU']:.4f}")


In [ ]:
# 10) TensorBoard (в отдельном терминале на сервере):
#   tensorboard --logdir runs --bind_all --port 6006
print("TB logs:", Path(cfg["train"]["runs_dir"]) / cfg["experiment_name"] / "tb")